# PKE - Wymiana klucza Diffiego-Hellmana

## Problem logarytmu dyskretnego

Niech $(G,\circ)$ będzie grupą z działaniem $\circ$ i elementem neutralnym $1_G$. Wtedy dla dowolnego elementu $a\in G$ i $k\in\mathbb{Z}$ definiujemy *potęgę* $$a^k =\left\{\begin{array}{cc}
\underbrace{a\circ a\circ \ldots \circ a}_{k}&\text{ dla }k>0\\
1_G&\text{ dla }k=0\\
\underbrace{a^{-1}\circ a^{-1}\circ \ldots \circ a^{-1}}_{k}&\text{ dla }k<0
\end{array}\right.$$
gdzie $a^{-1}$ jest elementem odwrotnym do $a$.

Dla $a,b\in G$, $b\neq 1_G$, *logarytmem dyskretnym* $\log_b a$ jest każda liczba $k\in\mathbb{Z}$ taka, że $b^k=a$.

## Logarytm dyskretny w $\mathbb{Z}_n$

W przypadku pierścienia $\mathbb{Z}_n$ logarytmem dyskretnym $\log_b a$ jest każda liczba $k\in\mathbb{Z}$ taka, że $b^k=a\mod n$, o ile w ogóle istnieje.

Specyficzną sytuacją w teorii liczb jest gdy $n=p$ jest liczbą pierwszą a $q$ jest pierwiastkiem pierwotnym $\mod p$. Wtedy:
- potęgi $q^k\mod p$ generują cały zbiór $[1,p-1]$, tzn. $q$ jest generatorem grupy multiplikatywnej rzędu $p-1$
- logarytm dyskretny $\log_q a$ istnieje dla każdego niezerowego elementu $a\in \mathbb{Z}_p$

## Algorytm baby-step giant-step

Jeden z najprostszych (poza metodą naiwną) algorytmów poszukiwania logarytmu dyskretnego w grupach cyklicznych.

Niech $p$ będzie liczbą pierwszą oraz niech $q$ będzie pierwiastkiem pierwotnym modulo $p$. Dla niezerowego $a\in\mathbb{Z}_p$ szukamy liczby $k\in\mathbb{Z}$ takiej, że $q^k=a\mod p$

### Krok 1.
- $m=\lceil\sqrt{p-1}\rceil$
- tworzymy pomocniczą tablicę potęg: dla wszystkich $i\in [0,m)$ obliczamy parę $(i,q^i)$
- obliczamy $r=(q^{-1})^m$
### Krok 2.
- $b=a$
- dla wszystkich $j\in [0,m)$:
    - sprawdzamy, czy para $(i,b)$ jest elementem tablicy potęg dla pewnego $i$
    - jeżeli tak, to $k=jm+i$ i kończymy algorytm
    - jeżeli nie, to $b=br$ i kontynuujemy pętlę


In [1]:
import math
import time

## Zadanie 1.

Zaimplementować algorytm baby-step giant-step. Przetestować dla podanych danych testujących.

```Dane testujące:
p = 7
q = 3
a = 4

m = 3
tablica_testowa = [1,3,2]
r = 6
k = 4 (j = 1, i = 1)
```

```
p = 29
q = 8
a = 10

m = 6
tablica_testowa = [1,8,6,19,7,27]
r = 9
k = 17 (j = 2, i = 5)
```

```
p = 113
q = 76
a = 84

m = 11
tablica_testowa = [1,76,13,84,56,75,50,71,85,19,88]
r = 70
k = 3 (j = 0, i = 3)
```

In [2]:
def baby_step_giant_step(p, q, a):
    m = math.ceil(math.sqrt(p - 1))
    
    powers_table = {}
    for i in range(m):
        value = pow(q, i, p)
        powers_table[value] = i
        
    q_inv = pow(q, p - 2, p) 
    r = pow(q_inv, m, p)
    
    b = a
    for j in range(m):
        if b in powers_table:
            i = powers_table[b]
            k = j * m + i
            return k
        
        b = (b * r) % p
        
    return None

In [3]:
test_data = [
    {"p": 7, "q": 3, "a": 4, "expected_k": 4},
    {"p": 29, "q": 8, "a": 10, "expected_k": 17},
    {"p": 113, "q": 76, "a": 84, "expected_k": 3}
]

print("Baby-Step Giant-Step Algorithm")
for test in test_data:
    p = test["p"]
    q = test["q"]
    a = test["a"]
    expected = test["expected_k"]
    
    result_k = baby_step_giant_step(p, q, a)
    print(f"p = {p:>3}, q = {q:>2}, a = {a:>2} | Found k = {result_k:>2} | (Expected k = {expected:>2})")
    assert result_k == expected, f"Error for p={p}, q={q}, a={a}"

Baby-Step Giant-Step Algorithm
p =   7, q =  3, a =  4 | Found k =  4 | (Expected k =  4)
p =  29, q =  8, a = 10 | Found k = 17 | (Expected k = 17)
p = 113, q = 76, a = 84 | Found k =  3 | (Expected k =  3)


## Algorytm wymiany klucza Diffiego-Hellmana

Alice i Bob uzgadniają klucz publiczny będący liczbą pierwszą $p$ oraz $q$ - pierwiastkiem pierwotnym mod $p$.
- sekret Alice: liczba całkowita $n\in \mathbb{Z}_p\setminus\{0\}$
- sekret Boba: liczba całkowita $m\in \mathbb{Z}_p\setminus\{0\}$
- Alice generuje $x=q^n\mod p$ i wysyła do Boba
- Bob generuje $y=q^m\mod p$ i wysyła Alice
- Alice oblicza klucz $k=y^n\mod p$
- Bob oblicza klucz $k=x^m\mod p$


## Zadanie 2.

Zaimplementuj powyższy algorytm wymiany klucza.

In [4]:
def diffie_hellman_key_exchange(p, q, n, m):
    print("Public Parameters")
    print(f"  p (prime number) = {p}")
    print(f"  q (primitive root) = {q}\n")

    print("Secrets")
    print(f"  Alice's secret (n) = {n}")
    print(f"  Bob's secret (m) = {m}\n")

    x = pow(q, n, p)
    print(f"Alice generates and sends x = {x}")

    y = pow(q, m, p)
    print(f"Bob generates and sends y = {y}\n")

    k_alice = pow(y, n, p)
    print(f"Alice calculates the shared key: k = {k_alice}")

    k_bob = pow(x, m, p)
    print(f"Bob calculates the shared key: k = {k_bob}\n")

    assert k_alice == k_bob, "Error! The calculated keys are not identical."
    print("Alice and Bob have generated an identical secret key.")
    
    return k_alice

In [5]:
p = 113
q = 76

# Random secrets n and m from the set [1, p-1]
alice_secret = 14
bob_secret = 89

shared_key = diffie_hellman_key_exchange(p, q, alice_secret, bob_secret)

Public Parameters
  p (prime number) = 113
  q (primitive root) = 76

Secrets
  Alice's secret (n) = 14
  Bob's secret (m) = 89

Alice generates and sends x = 69
Bob generates and sends y = 5

Alice calculates the shared key: k = 69
Bob calculates the shared key: k = 69

Alice and Bob have generated an identical secret key.


## Zadanie 3.
Na swoją własną implementację przeprowadź atak algorytmem baby-step, giant-step. Jeżeli atak zakónczył się sukcesem, zmodyfikuj parametry $p$ i $q$ tak, żeby atak algorytmem z zadania 1 nie zakończył się sukcesem w rozsądnym czasie.

In [6]:
print("BSGS Attack on old parameters")

p_small = 113
q_small = 76
n_alice = 14
m_bob = 89

x_public = pow(q_small, n_alice, p_small)
y_public = pow(q_small, m_bob, p_small)

print(f"Eve intercepts: p={p_small}, q={q_small}, x={x_public}, y={y_public}\n")

print("Running the Baby-Step Giant-Step algorithm...")
start_time = time.time()
cracked_secret_n = baby_step_giant_step(p_small, q_small, x_public)
end_time = time.time()

if cracked_secret_n is not None:
    print(f"Attack Successful! Eve found Alice's secret (n) = {cracked_secret_n}")
    print(f"Attack duration: {end_time - start_time:.5f} seconds")
    
    stolen_key = pow(y_public, cracked_secret_n, p_small)
    print(f"Eve generates the secret shared key: k = {stolen_key}\n")
else:
    print("Attack failed.")

BSGS Attack on old parameters
Eve intercepts: p=113, q=76, x=69, y=5

Running the Baby-Step Giant-Step algorithm...
Attack Successful! Eve found Alice's secret (n) = 14
Attack duration: 0.00002 seconds
Eve generates the secret shared key: k = 69



In [7]:
print("Key Exchange with Secure Parameters")

p_secure = 2**127 - 1 
q_secure = 3  

alice_secure_secret = 1234567890123456789  # Alice's new secret
bob_secure_secret = 9876543210987654321    # Bob's new secret

print("Exchanging keys for a 127-bit p...")
x_new = pow(q_secure, alice_secure_secret, p_secure)
y_new = pow(q_secure, bob_secure_secret, p_secure)

k_alice = pow(y_new, alice_secure_secret, p_secure)
k_bob = pow(x_new, bob_secure_secret, p_secure)

assert k_alice == k_bob
print("Users generated a new key!\n")

m_required = math.ceil(math.sqrt(p_secure - 1))
print(f"The required size of the 'baby-steps' table (m) is approximately: {m_required}")

Key Exchange with Secure Parameters
Exchanging keys for a 127-bit p...
Users generated a new key!

The required size of the 'baby-steps' table (m) is approximately: 13043817825332783104
